Am Ende ist es nicht schlimm, wenn die Vorhersage nicht funktioniert, ich sollte aber
verschiedene Wege prüfen und evaluieren. Und dann begründen warum es nicht funktioniert hat.
"Es ist mehr Wert, wenn das Modell nicht (direkt) funktioniert und man verschiedene Wege durchgeht und evaluiert,
als wenn das Modell direkt funktioniert."
Alles in den Notebooks dokumentieren und festhalten. Dem Leser soll klar werden wo seine Zeit investiert wurde.

#Verschiedene Wege zum Evaluieren:
1. fewshot prompting
2. loss function for unbalanced data => welche?
3. LLM Ansatz (beweisen, dass es auf LLMs geht mit Prompt bezüglich der Aufgabe: bsp.: Sage die interruptions hervor. Begründen, warum es bei BERT nicht geht. Begründung (kann 1 zu 1 ins Notebook übernommen werden): Es liegt daran, dass die Trainingsdaten in 0en und 1en umgewandet werden. Die interruptions kommen und ca. alle 10 Sätze vor und dadurch sind die Trainingsdaten unbalanced. Hat auch was mit der Cross Entropy Loss Function zutun. ChatGPT fragen.)
- llm mit llama cpp auf google colab möglich?
4. weitere überlegen

In [1]:
! pip install datasets
! pip install torch
! pip install evaluate
!pip install optuna transformers torch

! pip install transformers==4.28.1
! pip install accelerate==0.15.0
! pip install tokenizers==0.13.3
!pip install --upgrade google-auth-oauthlib google-auth-httplib2 google-api-python-client google-auth

  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.1/380.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.4/233.4 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 53.1 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transform

In [1]:
import json
import torch
import re
from transformers import BertTokenizer, BertForTokenClassification, Trainer, TrainingArguments
from transformers.modeling_outputs import TokenClassifierOutput
from torch.utils.data import Dataset, random_split
import torch.nn as nn
import torch.nn.functional as F

In [1]:
# Google Drive einbinden
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# import os
# current_directory = os.getcwd()

# files = os.listdir(current_directory)

# print("Current Directory:", current_directory)
# print("Files in the Directory:", files)


Current Directory: /content
Files in the Directory: ['.config', 'drive', 'sample_data']


In [8]:
# Vorverarbeitung durch regex (Zurufe mit Namen und ohne Namen)
#remove_parentheses_pattern = r'\(.*?\)'
#interruption_within_parentheses_pattern = r'\([^)]*<interruption>[^)]*\)'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Laden der Daten
file_path = '/content/drive/My Drive/Colab_Notebooks/modified_texts_with_interruption.json'
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Begrenzen Sie die Daten auf die ersten 1000 Einträge
data = data[:10000]

# Extrahieren der Texte und Labels aus dem JSON-Datensatz
texts = []
labels = []

for entry in data:
    text = entry["modified_text"]  # Verwenden Sie den modifizierten Text mit den bereits vorhandenen <interruption> Tokens
    texts.append(text)

    # Erstellen Sie Labels für jedes Token im Text (1 für <interruption> Token, 0 für andere)
    label = []
    tokens = text.split()
    for token in tokens:
        if "<interruption>" in token:
            label.append(1)
        else:
            label.append(0)
    labels.append(label)

# Padding der Labels
max_len = 512
padded_labels = [label + [0] * (max_len - len(label)) if len(label) < max_len else label[:max_len] for label in labels]

# Beispiel zur Veranschaulichung der Label-Generierung
example_texts = texts[:5]  # Nehmen wir die ersten 5 Texte
example_labels = labels[:5]

for i, text in enumerate(example_texts):
    print(f"Text {i+1}: {text}")
    print(f"Labels {i+1}: {example_labels[i]}")

# Beispiel zur Veranschaulichung der gepaddeten Labels
for i, label in enumerate(padded_labels[:5]):
    print(f"Padded Labels {i+1}: {label}")


Text 1: Sehr geehrter Herr Präsident! Sehr geehrte Damen und Herren! Verehrte Bürger! In diesen Coronazeiten – wir haben es in dieser Woche schon öfter gehört – ist wenig normal. Die Bewältigung der Coronakrise hat erhebliche Auswirkungen auf den Bundeshaushalt.
Besonders davon betroffen ist – kein Wunder – der Bereich des Einzelplans 11, Arbeit und Soziales. Der Haushaltsentwurf 2021 für das Bundesministerium für Arbeit und Soziales hat einen Umfang von insgesamt rund 165 Milliarden Euro und ist damit wieder der größte Einzelplan im Bundeshaushalt. Vom Entwurf bis zur Bereinigungssitzung wuchs dieser Haushaltsplan auf knapp 1 Milliarde Euro auf.
Ein Teil dieses Aufwuchses ist nachvollziehbar, da sich einige Zuschüsse und Leistungen an die Herbstprojektion und die Steuerschätzung anlehnen und dadurch zur Bereinigungssitzung angepasst werden. Was mich aber ärgert, sind neue Maßnahmen, die erst zur Bereinigungssitzung auftauchen und für die dann neue Mittel beantragt werden. Im letzten J

In [9]:
#Ein benutzerdefiniertes Dataset erstellen
class InterruptionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=False,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        labels = torch.tensor(labels[:self.max_len] + [0] * (self.max_len - len(labels)), dtype=torch.long)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }


tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
dataset = InterruptionDataset(texts, padded_labels, tokenizer, max_len=max_len)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, val_size])


In [10]:
# Modell mit Dropout und Gewichtsnormierung
class CustomBertForTokenClassification(BertForTokenClassification):
    def __init__(self, config):
        super().__init__(config)
        self.dropout = nn.Dropout(p=0.5)
        self.classifier = nn.utils.weight_norm(nn.Linear(config.hidden_size, config.num_labels))

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, position_ids=None, head_mask=None, inputs_embeds=None, labels=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, position_ids=position_ids, head_mask=head_mask, inputs_embeds=inputs_embeds, output_attentions=output_attentions, output_hidden_states=output_hidden_states, return_dict=return_dict)
        sequence_output = self.dropout(outputs[0])
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss() #implizite Berechnung der Softmax
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return TokenClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions)

model = CustomBertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2).to(device)


Some weights of the model checkpoint at bert-base-uncased were not used when initializing CustomBertForTokenClassification: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing CustomBertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CustomBertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of CustomBertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and

In [11]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",  #
    load_best_model_at_end=True,
    fp16=True  # Mixed precision training
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)


trainer.train()


Epoch,Training Loss,Validation Loss
1,0.028700,0.015069
2,0.014700,0.014681
3,0.014000,0.014169


TrainOutput(global_step=3000, training_loss=0.01913131586710612, metrics={'train_runtime': 1646.4419, 'train_samples_per_second': 14.577, 'train_steps_per_second': 1.822, 'total_flos': 6271122309120000.0, 'train_loss': 0.01913131586710612, 'epoch': 3.0})

In [12]:
# speichern des Modells nach dem Training, sodass es wiederverwendet werden kann, ohne es neu trainieren zu müssen
model.save_pretrained('./saved_model')
tokenizer.save_pretrained('./saved_model')

('./saved_model/tokenizer_config.json',
 './saved_model/special_tokens_map.json',
 './saved_model/vocab.txt',
 './saved_model/added_tokens.json')

In [13]:

results = trainer.evaluate()
print(results)


{'eval_loss': 0.014169445261359215, 'eval_runtime': 53.8461, 'eval_samples_per_second': 37.143, 'eval_steps_per_second': 4.643, 'epoch': 3.0}


In [20]:

model = BertForTokenClassification.from_pretrained('./saved_model', num_labels=2).to(device)

# Vorhersagen auf neuen Texten ohne <interruption> Token
test_texts = ["Sehr geehrte Damen und Herren, heute möchte ich über ein wichtiges Thema sprechen. Entschuldigung, darf ich kurz unterbrechen? <interruption> Es geht um die aktuellen wirtschaftlichen Entwicklungen. Die Zahlen zeigen einen positiven Trend, aber wir müssen vorsichtig sein. Ich habe eine Frage zu den genauen Daten, die Sie erwähnt haben. Wir dürfen nicht vergessen, dass viele Menschen noch immer von der Krise betroffen sind. Es ist entscheidend, dass wir weiterhin Maßnahmen zur Unterstützung anbieten. Vielen Dank für Ihre Aufmerksamkeit."]
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
test_encodings = {key: val.to(device) for key, val in test_encodings.items()}  # Daten auf die GPU verschieben
outputs = model(**test_encodings)
predictions = torch.argmax(outputs.logits, dim=-1)


print(predictions.cpu().numpy())


Some weights of the model checkpoint at ./saved_model were not used when initializing BertForTokenClassification: ['classifier.weight_v', 'classifier.weight_g']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./saved_model and are newly initialized: ['classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[[0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1
  1 1 0 1 1 1 0 1 0 0 1 1 1 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 0 0 1 1 1 1 1 0
  0 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 0 1 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0
  0 0 0 0 0 0 1 1 0 0 0 0 0 1 1 1 1 1 0 1 1 0 0 1 1 1 1 1 0 1 0 0 0 1 0 0
  0 1 0 1 1 1 1 0 0 0 0 0 0 1]]


In [22]:
import torch
from transformers import BertTokenizer, BertForTokenClassification

# Initialisieren des Modells und Tokenizers
model = BertForTokenClassification.from_pretrained('./saved_model', num_labels=2).to(device)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Vorhersagen auf neuen Texten
test_texts = ["Sehr geehrte Damen und Herren, heute möchte ich über ein wichtiges Thema sprechen. Entschuldigung, darf ich kurz unterbrechen? Es geht um die aktuellen wirtschaftlichen Entwicklungen. Die Zahlen zeigen einen positiven Trend, aber wir müssen vorsichtig sein. Ich habe eine Frage zu den genauen Daten, die Sie erwähnt haben. Wir dürfen nicht vergessen, dass viele Menschen noch immer von der Krise betroffen sind. Es ist entscheidend, dass wir weiterhin Maßnahmen zur Unterstützung anbieten. Vielen Dank für Ihre Aufmerksamkeit."]
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
test_encodings = {key: val.to(device) for key, val in test_encodings.items()}  # Daten auf die GPU verschieben

outputs = model(**test_encodings)
predictions = torch.argmax(outputs.logits, dim=-1)

tokens = tokenizer.convert_ids_to_tokens(test_encodings['input_ids'][0])
predictions = predictions[0].cpu().numpy()

# Formatierung der Ausgabe
formatted_output = ""
for i, (token, prediction) in enumerate(zip(tokens, predictions)):
    if i == 0:
        formatted_output += f"Sentence {i+1}:\n"
    formatted_output += f"{token}: {prediction}\n"
    if token == '[SEP]':
        formatted_output += "\n"

print(formatted_output)

print(outputs.logits)



Some weights of the model checkpoint at ./saved_model were not used when initializing BertForTokenClassification: ['classifier.weight_v', 'classifier.weight_g']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./saved_model and are newly initialized: ['classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: Future

Sentence 1:
[CLS]: 0
se: 0
##hr: 1
gee: 1
##hr: 1
##te: 1
dame: 0
##n: 0
und: 0
herr: 0
##en: 0
,: 0
he: 1
##ute: 1
mo: 0
##cht: 0
##e: 0
ich: 0
uber: 0
ein: 0
wi: 0
##cht: 0
##ige: 0
##s: 0
them: 0
##a: 0
sp: 1
##re: 1
##chen: 1
.: 1
en: 1
##ts: 0
##chu: 1
##ld: 0
##ig: 1
##ung: 1
,: 1
dar: 1
##f: 1
ich: 1
ku: 1
##rz: 1
un: 1
##ter: 1
##bre: 1
##chen: 1
?: 1
es: 1
ge: 1
##ht: 1
um: 1
die: 1
ak: 1
##tu: 1
##elle: 1
##n: 1
wi: 1
##rts: 1
##chaft: 1
##liche: 1
##n: 1
en: 1
##t: 1
##wick: 1
##lun: 1
##gen: 1
.: 1
die: 1
za: 1
##hl: 1
##en: 1
ze: 1
##igen: 1
eine: 0
##n: 1
positive: 0
##n: 1
trend: 1
,: 1
abe: 1
##r: 1
wi: 1
##r: 1
mu: 1
##ssen: 1
vo: 1
##rs: 1
##ich: 1
##ti: 1
##g: 1
se: 1
##in: 1
.: 1
ich: 1
ha: 1
##be: 1
eine: 1
fra: 1
##ge: 1
zu: 1
den: 1
gen: 1
##au: 1
##en: 1
date: 1
##n: 1
,: 1
die: 1
si: 1
##e: 1
er: 1
##wa: 1
##hn: 1
##t: 1
ha: 1
##ben: 1
.: 1
wi: 1
##r: 1
du: 1
##rf: 0
##en: 1
nic: 1
##ht: 1
verge: 1
##ssen: 1
,: 1
das: 1
##s: 1
vie: 1
##le: 1
men: 1
##schen: 1
n